# Next Word Prediction using Hamlet

Corrected version of the LSTM next-word prediction code.

In [ ]:
import nltk
import numpy as np
import tensorflow as tf

from nltk.corpus import gutenberg
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense, Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

In [ ]:
nltk.download('gutenberg')

text = gutenberg.raw('shakespeare-hamlet.txt')

with open('hamlet.txt', 'w', encoding='utf-8') as f:
    f.write(text)

print(text[:300])

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

# Add 1 because tokenizer word indexes start from 1. Index 0 is reserved for padding.
total_words = len(tokenizer.word_index) + 1

print('Total words:', total_words)

In [ ]:
input_sequences = []

for line in text.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]

    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i + 1]
        input_sequences.append(n_gram_sequence)

max_seq_len = max(len(x) for x in input_sequences)
max_seq_len

In [ ]:
input_sequences = np.array(
    pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')
)

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

y = tf.keras.utils.to_categorical(y, num_classes=total_words)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, y_train.shape)

In [ ]:
model = Sequential()
model.add(Embedding(total_words, 100, input_length=X_train.shape[1]))
model.add(LSTM(150, return_sequences=True))
model.add(Dropout(0.2))

# The final LSTM must return one vector because y has one next-word label per row.
model.add(LSTM(100))
model.add(Dense(total_words, activation='softmax'))

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    verbose=1
)

In [ ]:
def predict_next_word(seed_text):
    token_list = tokenizer.texts_to_sequences([seed_text])[0]
    token_list = pad_sequences([token_list], maxlen=max_seq_len - 1, padding='pre')

    predicted_index = np.argmax(model.predict(token_list, verbose=0), axis=-1)[0]

    for word, index in tokenizer.word_index.items():
        if index == predicted_index:
            return word

    return None

predict_next_word('to be')